# Alien Physics Discovery — OpenEnv Hackathon 2026
### Train an LLM to do unsupervised scientific discovery from raw sensor data

**Theme:** Wild Card (Theme 5) + World Modeling (Theme 3.1) + Self-Improvement (Theme 4)

**The Problem:** LLMs can write code and explain physics, but have never been trained to
*discover* physical structure from raw, ambiguous time-series data through iterative theory revision.

**The Environment:** An agent observes noisy sensor logs from a simulated alien universe
governed by hidden ODEs. It must write a Python `Theory` class that models the dynamics.
The environment executes the code against future observations. Reward = log-likelihood − MDL penalty.

---
> ⚡ **Runtime:** T4 GPU (free Colab tier works)  
> 📦 **Stack:** Unsloth + HF TRL (GRPO) + project's sandboxed execution

## Step 1 — Install Dependencies

In [ ]:
%%capture
# Unsloth must come first — it patches transformers in-place
!pip install "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install "trl>=0.9.0" peft accelerate bitsandbytes datasets wandb huggingface_hub
!pip install matplotlib numpy scipy

# Install the Ontology Architect environment from the repo
# Replace with your actual repo URL before submitting
!pip install "git+https://github.com/rohan-27p/ontology_architect.git"

print('All dependencies installed')

## Step 2 — Imports & Config

In [ ]:
import json, os, re, warnings
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# ── Project imports ────────────────────────────────────────────────────────
from ontology_architect.baselines import get_baseline
from ontology_architect.config import (
    ExperimentConfig, FeedbackConfig, RewardConfig, SandboxConfig, UniverseConfig
)
from ontology_architect.models import OntologyArchitectAction
from ontology_architect.sandbox import TheorySandbox
from ontology_architect.reward import score_theory
from ontology_architect.server.ontology_architect_environment import OntologyArchitectEnvironment
from ontology_architect.universe import ProceduralAlienUniverse

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Model — Qwen2.5-Coder-1.5B fits T4 (16 GB) comfortably ───────────────
MODEL_NAME  = 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit'
MAX_SEQ_LEN = 2048

# ── Training knobs ─────────────────────────────────────────────────────────
NUM_EPISODES   = 60    # oracle curriculum episodes
TRAIN_EPOCHS   = 2
NUM_GENS       = 4     # GRPO group size — 2 keeps T4 safe
MAX_COMPLETION = 768   # theory code tokens
BATCH_SIZE     = 1
GRAD_ACCUM     = 8
LR             = 2e-5
USE_WANDB      = False
HF_TOKEN       = ''    # set to push model to Hub

# ── Environment config — matches tiny_smoke but longer for richer signal ──
ENV_CONFIG = ExperimentConfig(
    universe=UniverseConfig(
        seed=SEED,
        family='dual_fluid',
        observation_window=12,
        future_window=4,
        max_steps=4,
        anomaly_rate=0.06,
        drift_interval=6,
    ),
    sandbox=SandboxConfig(mode='subprocess', timeout_seconds=4.0),
    reward=RewardConfig(
        mdl_lambda=0.0005,
        prediction_sigma=0.12,
        anomaly_bonus=2.0,
        drift_bonus=1.0,
        execution_error_penalty=-25.0,
    ),
    feedback=FeedbackConfig(peer_review_window=4, lineage_window=4),
)

print(f'Config ready')
print(f'  Model   : {MODEL_NAME}')
print(f'  Device  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'  VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

## Step 3 — Generate Oracle Curriculum

The teacher baseline rolls out `NUM_EPISODES` episodes. Each step saves `(prompt, completion, reward, future_records)`. The `future_records` are the ground-truth sensor windows — we store them so the GRPO reward function can score any new theory against the exact same future without re-running the environment.

In [ ]:
CURRICULUM_PATH = Path('artifacts/curriculum/grpo_curriculum.jsonl')
CURRICULUM_PATH.parent.mkdir(parents=True, exist_ok=True)

FAMILIES = ['dual_fluid', 'coupled_oscillator']
teacher  = get_baseline('teacher')
sandbox  = TheorySandbox(ENV_CONFIG.sandbox)

examples = []

with CURRICULUM_PATH.open('w') as fh:
    for ep in range(NUM_EPISODES):
        # Rotate through families so the model sees both universes
        family = FAMILIES[ep % len(FAMILIES)]
        cfg = ExperimentConfig(
            universe=UniverseConfig(
                seed=SEED + ep * 997,
                family=family,
                observation_window=ENV_CONFIG.universe.observation_window,
                future_window=ENV_CONFIG.universe.future_window,
                max_steps=ENV_CONFIG.universe.max_steps,
                anomaly_rate=ENV_CONFIG.universe.anomaly_rate,
                drift_interval=ENV_CONFIG.universe.drift_interval,
            ),
            sandbox=ENV_CONFIG.sandbox,
            reward=ENV_CONFIG.reward,
            feedback=ENV_CONFIG.feedback,
        )
        env = OntologyArchitectEnvironment(cfg)
        obs = env.reset()

        for step in range(cfg.universe.max_steps):
            # Capture the future window that this step will be scored against
            cursor = env._cursor
            future_records = env._records[cursor : cursor + cfg.universe.future_window]
            sensor_names   = list(env._universe.sensor_names)

            row = {
                'prompt'        : obs.text,
                'completion'    : teacher,
                'episode'       : ep,
                'step'          : step,
                'family'        : family,
                'sensor_names'  : sensor_names,
                # Serialise future so reward_fn can use it without re-running env
                'future_json'   : json.dumps([
                    {'t': r.t, 'sensors': r.sensors, 'anomaly': r.anomaly, 'drift': r.drift}
                    for r in future_records
                ]),
            }

            obs = env.step(OntologyArchitectAction(
                theory_module=teacher,
                revision_note=f'curriculum ep={ep} step={step}',
            ))
            row['reward'] = obs.reward
            row['metrics'] = json.dumps(obs.metadata.get('last_metrics', {}))

            fh.write(json.dumps(row) + '\n')
            examples.append(row)

            if obs.done:
                break

print(f'Generated {len(examples)} curriculum examples across {NUM_EPISODES} episodes')
print(f'  Families: {set(e["family"] for e in examples)}')
print(f'  Reward range: [{min(e["reward"] for e in examples):.3f}, {max(e["reward"] for e in examples):.3f}]')
teacher_mean = float(np.mean([e['reward'] for e in examples]))
print(f'  Teacher mean reward: {teacher_mean:.4f}  (this is the bar GRPO must beat)')

## Step 4 — Build HuggingFace Dataset

In [ ]:
SYSTEM_PROMPT = """You are a scientific discovery agent. An alien universe is governed by hidden physical laws.
You observe noisy sensor logs and must write a compact Python Theory class that predicts future readings.

Your class MUST implement:
  fit(history)            — history is a list of dicts: [{'sensors': {'sigma': float, ...}}, ...]
  predict(window)         — window is {'steps': int, 'history': [...], 'sensor_names': [...]}
                            return a list of {'sensors': {...}, 'anomaly_prob': float}
  log_prob(observations)  — return a float

Optional but rewarded:
  detect_drift(history) -> bool
  describe()            -> str

Allowed imports: numpy, scipy, math, statistics, collections, typing, random, functools, itertools
DO NOT import anything else. DO NOT use file I/O, network, or system calls.
Shorter correct theories score higher than longer ones (MDL principle).
Reply with ONLY the Python code — no markdown fences, no explanation."""


def make_chat_prompt(obs_text: str, tokenizer) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': obs_text},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


# We load raw rows first; prompts are formatted after the tokenizer is loaded
raw_rows = [json.loads(l) for l in CURRICULUM_PATH.read_text().splitlines() if l.strip()]
print(f'Loaded {len(raw_rows)} rows from curriculum')

## Step 5 — Load Model with Unsloth

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,       # auto: bf16 on Ampere+, fp16 otherwise
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    target_modules             = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                   'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha                 = 16,
    lora_dropout               = 0.05,
    bias                       = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state               = SEED,
)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model loaded: {total:,} total params, {trainable:,} trainable ({100*trainable/total:.2f}%)')

# Ensure pad token is set (needed for left-padding in GRPO)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

## Step 6 — Format Dataset with Chat Template

In [ ]:
dataset_rows = []
for row in raw_rows:
    dataset_rows.append({
        'prompt'      : make_chat_prompt(row['prompt'], tokenizer),
        'future_json' : row['future_json'],
        'history_json' : row.get('history_json', '[]'),
        'sensor_names': json.dumps(row['sensor_names']),
    })

dataset = Dataset.from_list(dataset_rows)
print(f'Dataset: {len(dataset)} rows')
print(f'Columns: {dataset.column_names}')
print(f'Sample prompt length: {len(dataset[0]["prompt"])} chars')

## Step 7 — GRPO Reward Function

Each completion is scored by running it through the project's real sandbox and scorer against
the **pre-captured future window** stored in the dataset row. This avoids the stale-trajectory
bug in the original notebook where the reward used a separately-seeded universe.

In [ ]:
from ontology_architect.universe import SensorRecord

_sandbox = TheorySandbox(ENV_CONFIG.sandbox)
_reward_cfg = ENV_CONFIG.reward


def _deserialise_future(future_json: str) -> list[SensorRecord]:
    return [
        SensorRecord(t=r['t'], sensors=r['sensors'], anomaly=r['anomaly'], drift=r['drift'])
        for r in json.loads(future_json)
    ]


def physics_reward_fn(
    prompts     : list[str],
    completions : list[str],
    future_json : list[str],
    history_json: list[str],
    sensor_names: list[str],
    **kwargs,
) -> list[float]:
    """
    GRPO reward function.  Columns `future_json` and `sensor_names` are
    passed automatically by GRPOTrainer from the dataset.
    """
    rewards = []
    for theory_code, fj, hj_str, sn_json in zip(completions, future_json, history_json, sensor_names):
        try:
            # Strip markdown fences the model sometimes emits
            theory_code = re.sub(r'^```(?:python)?\s*', '', theory_code.strip())
            theory_code = re.sub(r'```\s*$', '', theory_code)

            future       = _deserialise_future(fj)
            names        = tuple(json.loads(sn_json))
            history_data = json.loads(hj_str) if hj_str else []
            sandbox_res  = _sandbox.execute(theory_code, history_data, future, names)
            breakdown    = score_theory(sandbox_res, future, theory_code, _reward_cfg)
            reward       = float(np.clip(breakdown.reward, -30.0, 5.0))
        except Exception:
            reward = float(_reward_cfg.execution_error_penalty)
        rewards.append(reward)
    return rewards


# Sanity check — teacher should score positively
row0 = dataset_rows[0]
test_rewards = physics_reward_fn(
    prompts      = [row0['prompt']],
    completions  = [get_baseline('teacher')],
    future_json  = [row0['future_json']],
    sensor_names = [row0['sensor_names']],
)
print(f'Sanity check — teacher reward on row 0: {test_rewards[0]:.4f}')
print(f'  (Expected roughly > {teacher_mean:.2f} for a good theory)')

## Step 8 — GRPO Training

In [ ]:
if USE_WANDB:
    import wandb
    wandb.init(project='alien-physics-discovery', name='qwen2.5-coder-1.5b-grpo')

grpo_config = GRPOConfig(
    output_dir                  = 'artifacts/checkpoints/grpo',
    num_train_epochs            = TRAIN_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    warmup_ratio                = 0.05,
    lr_scheduler_type           = 'cosine',
    num_generations             = NUM_GENS,
    max_completion_length       = MAX_COMPLETION,   # correct TRL >=0.9 field
    logging_steps               = 5,
    save_steps                  = 50,
    save_total_limit            = 2,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    report_to                   = 'wandb' if USE_WANDB else 'none',
    beta                        = 0.001,
    max_grad_norm               = 0.3,
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    args             = grpo_config,          # 'args', not 'config'
    train_dataset    = dataset,
    reward_funcs     = [physics_reward_fn],
)

print('Starting GRPO training')
print(f'  Model       : {MODEL_NAME}')
print(f'  Samples     : {len(dataset)}')
print(f'  Epochs      : {TRAIN_EPOCHS}')
print(f'  Group size  : {NUM_GENS}')
print(f'  Batch×accum : {BATCH_SIZE}×{GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} effective')

train_result = trainer.train()
print(f'\nTraining complete — final loss: {train_result.training_loss:.4f}')

## Step 9 — Plot Training Curves

In [ ]:
logs    = trainer.state.log_history
steps   = [l['step']           for l in logs if 'loss'   in l]
losses  = [l['loss']           for l in logs if 'loss'   in l]
r_pairs = [(l['step'], l['reward']) for l in logs if 'reward' in l]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Alien Physics Discovery — GRPO Training (Qwen2.5-Coder-1.5B)', fontweight='bold')

ax1.plot(steps, losses, color='#4f86c6', linewidth=2)
ax1.set_xlabel('Training Step')
ax1.set_ylabel('Loss')
ax1.set_title('GRPO Loss')
ax1.grid(alpha=0.3)

if r_pairs:
    rs, rv = zip(*r_pairs)
    ax2.plot(rs, rv, color='#e67e22', linewidth=2, label='Mean group reward')
    ax2.axhline(teacher_mean, color='gray', linestyle='--', alpha=0.7,
                label=f'Teacher baseline ({teacher_mean:.2f})')
    ax2.set_xlabel('Training Step')
    ax2.set_ylabel('MDL Reward')
    ax2.set_title('Reward vs Teacher Baseline')
    ax2.legend()
    ax2.grid(alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'Enable USE_WANDB for live\nreward curves, or check\nlog_history manually',
             ha='center', va='center', transform=ax2.transAxes, color='gray')

plt.tight_layout()
plt.savefig('artifacts/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved artifacts/training_curves.png')

## Step 10 — Before / After Evaluation

In [ ]:
FastLanguageModel.for_inference(model)

def generate_theory(obs_text: str, max_new: int = 480) -> str:
    prompt  = make_chat_prompt(obs_text, tokenizer)
    inputs  = tokenizer(prompt, return_tensors='pt',
                        truncation=True, max_length=MAX_SEQ_LEN - max_new).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens  = max_new,
            temperature     = 0.6,
            do_sample       = True,
            pad_token_id    = tokenizer.eos_token_id,
        )
    theory = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    theory = re.sub(r'^```(?:python)?\s*', '', theory.strip())
    theory = re.sub(r'```\s*$', '', theory)
    return theory


def eval_full_episode(family: str, seed: int) -> dict:
    from dataclasses import replace as dc_replace
    cfg = ExperimentConfig(
        universe=dc_replace(ENV_CONFIG.universe, family=family, seed=seed),
        sandbox=ENV_CONFIG.sandbox,
        reward=ENV_CONFIG.reward,
        feedback=ENV_CONFIG.feedback,
    )
    env  = OntologyArchitectEnvironment(cfg)
    obs  = env.reset()
    rewards, done = [], False
    final_theory  = ''
    while not done:
        theory = generate_theory(obs.text)
        obs    = env.step(OntologyArchitectAction(
            theory_module=theory, revision_note='eval'
        ))
        rewards.append(obs.reward)
        final_theory = theory
        done = obs.done
    return {'family': family, 'rewards': rewards,
            'mean_reward': float(np.mean(rewards)), 'final_theory': final_theory}


# Teacher baseline (untrained)
def eval_teacher_episode(family: str, seed: int) -> dict:
    from dataclasses import replace as dc_replace
    cfg = ExperimentConfig(
        universe=dc_replace(ENV_CONFIG.universe, family=family, seed=seed),
        sandbox=ENV_CONFIG.sandbox,
        reward=ENV_CONFIG.reward,
        feedback=ENV_CONFIG.feedback,
    )
    env     = OntologyArchitectEnvironment(cfg)
    obs     = env.reset()
    teacher = get_baseline('teacher')
    rewards, done = [], False
    while not done:
        obs  = env.step(OntologyArchitectAction(theory_module=teacher, revision_note='baseline'))
        rewards.append(obs.reward)
        done = obs.done
    return {'family': family, 'rewards': rewards, 'mean_reward': float(np.mean(rewards))}


print('Running eval episodes (trained model vs teacher baseline)...')
eval_seeds  = [9001, 9002]
eval_families = ['thermal_split', 'coupled_oscillator']

trained_results  = [eval_full_episode(f, s)    for f in eval_families for s in eval_seeds]
baseline_results = [eval_teacher_episode(f, s) for f in eval_families for s in eval_seeds]

trained_mean  = float(np.mean([r['mean_reward'] for r in trained_results]))
baseline_mean = float(np.mean([r['mean_reward'] for r in baseline_results]))

print(f'\nResults:')
print(f'  Trained model mean reward  : {trained_mean:.4f}')
print(f'  Teacher baseline mean reward: {baseline_mean:.4f}')
print(f'  Delta                       : {trained_mean - baseline_mean:+.4f}')

## Step 11 — Per-Turn Improvement Plot

In [ ]:
# Aggregate rewards by turn index across all eval episodes
max_turns = max(len(r['rewards']) for r in trained_results)

def pad_rewards(results, length):
    padded = [r['rewards'] + [r['rewards'][-1]] * (length - len(r['rewards'])) for r in results]
    return np.array(padded)

trained_mat  = pad_rewards(trained_results,  max_turns)
baseline_mat = pad_rewards(baseline_results, max_turns)
turns = list(range(1, max_turns + 1))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(turns, trained_mat.mean(0),  color='#3498db', marker='o', lw=2, label='Trained model')
ax.fill_between(turns,
                trained_mat.mean(0) - trained_mat.std(0),
                trained_mat.mean(0) + trained_mat.std(0),
                alpha=0.15, color='#3498db')
ax.plot(turns, baseline_mat.mean(0), color='gray',    marker='s', lw=2, linestyle='--', label='Teacher baseline')
ax.set_xlabel('Refinement Turn')
ax.set_ylabel('MDL Reward')
ax.set_title('Per-Turn Theory Improvement')
ax.set_xticks(turns)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('artifacts/per_turn_improvement.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved artifacts/per_turn_improvement.png')

## Step 12 — Show a Discovered Theory

In [ ]:
best = max(trained_results, key=lambda r: r['mean_reward'])
print(f"Best result — family={best['family']}  mean_reward={best['mean_reward']:.4f}")
print('=' * 70)
theory = best['final_theory']
print(theory[:2000])
if len(theory) > 2000:
    print(f'\n... [{len(theory)-2000} more chars] ...')

## Step 13 — Save Model & Push to Hub

In [ ]:
Path('artifacts').mkdir(exist_ok=True)
model.save_pretrained('artifacts/model')
tokenizer.save_pretrained('artifacts/model')
print('Model saved to artifacts/model')

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    model.push_to_hub('lostdecimal27/alien-physics-discovery-qwen2.5-1.5b')
    tokenizer.push_to_hub('lostdecimal27/alien-physics-discovery-qwen2.5-1.5b')
    print('Pushed to HuggingFace Hub')
else:
    print('Set HF_TOKEN above to push to Hub')

## Step 14 — Submission Checklist

In [ ]:
checks = {
    'artifacts/training_curves.png'    : Path('artifacts/training_curves.png').exists(),
    'artifacts/per_turn_improvement.png': Path('artifacts/per_turn_improvement.png').exists(),
    'artifacts/model (local save)'     : Path('artifacts/model').exists(),
    'openenv.yaml in repo'             : Path('openenv.yaml').exists(),
    'README.md in repo'                : Path('README.md').exists(),
    'Colab notebook (this file)'       : True,
}

print('OpenEnv Hackathon — Submission Checklist')
print('=' * 50)
for item, ok in checks.items():
    print(f"  {'✅' if ok else '❌'}  {item}")

print('\nRemaining manual steps:')
for todo in [
    '□  Push model to HuggingFace Hub (set HF_TOKEN)',
    '□  Deploy environment to HuggingFace Space',
    '□  Record < 2 min YouTube demo or write HF blog post',
    '□  Update lostdecimal27 placeholders above',
    '□  Update README with video/blog URL and HF Space URL',
    '□  Submit HF Space URL to hackathon portal',
]:
    print(f'  {todo}')

print(f'\nFinal score summary:')
print(f'  Teacher baseline mean reward : {baseline_mean:.4f}')
print(f'  Trained model mean reward    : {trained_mean:.4f}')
print(f'  Improvement                  : {trained_mean - baseline_mean:+.4f}')